# C4-classical-ml-practice — Practice p17 — Solution

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
FEATURES = ["temp_c", "vibration_mm_s", "pressure_pa"]
sensors = pd.read_csv("data/sensors.csv")
X = sensors[FEATURES].to_numpy()
y = sensors["status"].to_numpy()
acc_A = float(cross_val_score(KNeighborsClassifier(5), X, y, cv=5).mean())
X_leaky = StandardScaler().fit_transform(X)
acc_B = float(cross_val_score(KNeighborsClassifier(5), X_leaky, y, cv=5).mean())
honest_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(5)),
])
acc_C = float(cross_val_score(honest_pipe, X, y, cv=5).mean())

acc_A, acc_B, acc_C

A scores about 0.583 because the high-magnitude `pressure_pa` column dominates every raw fold’s distances even though pressure barely separates the classes. C’s 0.867 is the legitimate estimate: in B, each validation fold influenced the global scaling statistics before evaluation, and the small 0.017 gap cannot undo that information leak. The report should contain C’s mean, with the caveat that cross-validation estimates future performance from these sampled folds rather than guaranteeing it.

### Answer check

In [ ]:
assert np.isclose(acc_A, 7/12, atol=1e-9, rtol=0)
assert np.isclose(acc_B, 0.85, atol=1e-9, rtol=0)
assert np.isclose(acc_C, 13/15, atol=1e-9, rtol=0)
assert abs(acc_B - acc_C) < 0.02
assert acc_C > acc_B > acc_A